# ToxHabits NER

Clean companion notebook for CharCNN + BiLSTM + CRF training and error analysis.

In [ ]:
from pathlib import Path
import torch
import torch.optim as optim
from torch.utils.data import DataLoader

from toxhabits_ner_bilstm import (
    ENTITY_LABELS,
    ToxHabitsDataset,
    ToxHabitsNERWithCRF,
    build_bio_chunks,
    build_vocabularies,
    create_collate_fn,
    create_spanish_nlp,
    evaluate,
    fit,
    list_filenames,
    multilabel_train_val_test_split,
)
from toxhabits_ner_bilstm.error_analysis import analyze_errors, build_seen_entity_sets, get_predictions_with_metadata

In [ ]:
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
elif PROJECT_DIR.name != "03_toxhabits_ner_bilstm":
    PROJECT_DIR = PROJECT_DIR / "03_toxhabits_ner_bilstm"
ANNOTATION_DIR = PROJECT_DIR / "data" / "ToxNER" / "ToxHabits(ToxNER)_Train_ANNFiles" / "train_annotations"
MODEL_PATH = PROJECT_DIR / "models" / "tox_habits_ner_crf_model_base.pt"

nlp = create_spanish_nlp()
filenames = list_filenames(ANNOTATION_DIR)
train_files, val_files, test_files = multilabel_train_val_test_split(ANNOTATION_DIR, filenames)

train_chunks = [build_bio_chunks(ANNOTATION_DIR, filename, nlp, max_tokens=256) for filename in train_files]
val_chunks = [build_bio_chunks(ANNOTATION_DIR, filename, nlp, max_tokens=256) for filename in val_files]
test_chunks = [build_bio_chunks(ANNOTATION_DIR, filename, nlp, max_tokens=256) for filename in test_files]

In [ ]:
word2idx, char2idx, tag2idx = build_vocabularies(train_chunks)
idx2tag = {idx: tag for tag, idx in tag2idx.items()}
collate_fn = create_collate_fn(word2idx["<pad>"], char2idx["<pad>"], tag2idx["O"])

train_loader = DataLoader(ToxHabitsDataset(train_chunks, word2idx, char2idx, tag2idx), batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(ToxHabitsDataset(val_chunks, word2idx, char2idx, tag2idx), batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(ToxHabitsDataset(test_chunks, word2idx, char2idx, tag2idx), batch_size=16, shuffle=False, collate_fn=collate_fn)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ToxHabitsNERWithCRF(
    vocab_size=len(word2idx),
    word_pad_idx=word2idx["<pad>"],
    char_vocab_size=len(char2idx),
    char_pad_idx=char2idx["<pad>"],
    num_tags=len(tag2idx),
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
history, best_f1 = fit(model, train_loader, val_loader, optimizer, device, idx2tag, ENTITY_LABELS, epochs=30, patience=5)

In [ ]:
test_metrics = evaluate(model, test_loader, device, idx2tag, ENTITY_LABELS)
test_metrics

In [ ]:
train_rows = get_predictions_with_metadata(model, train_loader, device, idx2tag)
test_rows = get_predictions_with_metadata(model, test_loader, device, idx2tag)
seen_surface_forms, _ = build_seen_entity_sets(train_rows)
analysis = analyze_errors(test_rows, seen_surface_forms, ENTITY_LABELS)
analysis["summary_df"].round(4)